# Runtime Evaluation — Latency, Throughput, and the Tails That Break Real Systems

The first three notebooks in this series evaluate *what the estimator says*:
[trajectory analysis](trajectory_analysis.ipynb) and
[evaluation metrics](../vo_evaluation_metrics.ipynb) score the poses,
[estimator consistency](estimator_consistency.ipynb) scores the covariance, and
[benchmark methodology](benchmark_methodology.ipynb) makes the comparison
defensible. None of them ask *when* the estimator says it.

For anything running on a robot, that is half the specification. A method that
is 2× more accurate and 3× slower is **worse** on a drone, because the
controller now acts on state that is 3× more stale, and stale state on a fast
platform is worse than imprecise state. Accuracy and latency are not separable
axes to optimise independently — they trade against each other through the
control loop.

Runtime is also the axis where reported numbers are least trustworthy, because
the standard practice — one mean, one frequency, no platform description — hides
almost everything that matters.

**Companion docs:**

* [Trajectory analysis](trajectory_analysis.ipynb) · [Evaluation metrics](../vo_evaluation_metrics.ipynb) · [Estimator consistency](estimator_consistency.ipynb) · [Benchmark methodology](benchmark_methodology.ipynb)
* [Robotic system design](../robotic_system_design.md) — where the latency budget is actually spent
* [ROS 2 QoS](../ros2_qos.md) — queue depths and drop policy, which §4 depends on
* [Estimator parameter reference](../../vio_benchmark/docs/PARAMETERS.md)


## 1. Three numbers that get called "speed"

These are routinely conflated, and they are independent.

| Quantity | Definition | Who cares |
|---|---|---|
| **Throughput** | Sustained frames processed per second | Offline mapping, dataset reprocessing |
| **Latency** | Wall-clock time from a measurement's **timestamp** to the pose for that measurement being **available to the consumer** | Control loops, obstacle avoidance |
| **Real-time factor (RTF)** | wall clock elapsed / dataset duration. $\text{RTF} < 1$ = faster than real time | Whether you can run it live at all |

Latency is measured from the *sensor timestamp*, not from the start of your
`process()` call. It includes driver delay, transport, queueing, and any
buffering — all of which the algorithm's self-timing misses entirely.

### 1.1 Worked example — 4× the throughput, identical latency

A VO front-end with four stages, each costing 25 ms: undistort → detect →
match → solve.

**Single-threaded.** Each frame takes $4 \times 25 = 100$ ms.

$$
\text{throughput} = 10\ \text{Hz}, \qquad \text{latency} = 100\ \text{ms}
$$

**Pipelined across 4 cores**, one stage per core. Once the pipe is full, a
result emerges every 25 ms, but each individual frame still traverses all four
stages:

$$
\text{throughput} = 40\ \text{Hz}, \qquad \text{latency} = 100\ \text{ms}
$$

**Throughput improved 4×. Latency did not move at all.** The paper reports
"40 Hz, real-time" and it is true; the controller still acts on state that is
100 ms old. On a car at 15 m/s that is **1.5 m of travel** before the pose is
usable, and on a quadrotor it is often the difference between a stable and an
unstable attitude loop.

The corollary: **pipelining and batching improve the number that gets reported
and not the number that matters.** If a system quotes only a frequency, you
cannot tell whether its latency is 25 ms or 250 ms.


## 2. The mean is the wrong statistic — report the tail

VO/SLAM per-frame cost is strongly multi-modal by construction:

* **Tracking frames** — cheap, the common case.
* **Keyframe frames** — new feature extraction, triangulation, local bundle
  adjustment or marginalisation. 5–10× a tracking frame.
* **Loop-closure frames** — place recognition, geometric verification, pose-graph
  optimisation or global BA. 50–200×.

Averaging over that distribution produces a number that describes no frame that
actually occurred.

### 2.1 Worked example — "62 Hz" that stalls for 0.8 s

1000 frames: 950 tracking at 10 ms, 45 keyframes at 60 ms, 5 loop closures at
800 ms.

**Mean:**

$$
\frac{950(10) + 45(60) + 5(800)}{1000}
= \frac{9500 + 2700 + 4000}{1000}
= \frac{16200}{1000} = 16.2\ \text{ms} \;\Rightarrow\; \mathbf{61.7\ Hz}
$$

Reported as "62 Hz — 3× real-time headroom at 20 Hz."

**Percentiles** (nearest-rank on the 1000 sorted samples):

| Statistic | Value |
|---|---|
| p50 (500th) | 10 ms |
| p95 (950th) | **10 ms** |
| p99 (990th) | 60 ms |
| max | **800 ms** |

Note that **p95 is also useless here** — it is 10 ms, better than the mean, and
it hides the stall completely. The distribution's damage lives entirely above
p99. For a hard real-time loop the only statistic that matters is **max**.

**What the stall actually costs.** At a 20 Hz input rate the per-frame budget is
50 ms. An 800 ms frame overruns it by 750 ms, during which

$$
\frac{800\ \text{ms}}{50\ \text{ms}} = 16\ \text{frames}
$$

arrive with nowhere to go. Afterwards the pipeline drains at 10 ms per frame
while new ones still arrive every 50 ms — a net 4 frames cleared per 50 ms
period — so the backlog takes $16/4 = 4$ periods $\approx 200$ ms to clear.
Peak latency is ~800 ms and the disturbance lasts a full second, five times per
run.

And if the input queue is bounded — a ROS subscriber with `queue_size=5`, which
is typical — then **11 of those 16 frames are silently dropped**. Dropped frames
during a loop closure are precisely the frames the system most needed, and this
appears nowhere in the timing report. See [ROS 2 QoS](../ros2_qos.md).

**Report:** mean, p50, p99, max, and a histogram or cost-vs-frame plot. The
plot is the informative artefact — it shows the spikes co-located with the
events that caused them.


## 3. One number per system is not enough — report per-thread budgets

Modern VO/SLAM systems are not one loop. A typical architecture:

| Thread | Rate | Budget | Consequence of overrun |
|---|---|---|---|
| IMU propagation | 200 Hz | 5 ms | State output stops — fatal |
| Feature tracking / front-end | 20–30 Hz | 33–50 ms | Frames dropped, tracks lost |
| Local BA / marginalisation | ~keyframe rate | 100–300 ms | Map lags; tracking continues on stale map |
| Loop closure / global BA | seconds | seconds | Correction arrives late |

The architectural point: **moving expensive work off the critical path converts
a latency spike into map staleness**, which is a far cheaper failure. The §2.1
system, restructured so loop closure runs asynchronously, has a max front-end
cost of 60 ms instead of 800 ms — same total compute, completely different
real-time behaviour, and *identical* mean.

So a single "16.2 ms" describes neither system. Report a budget table: per
thread, its rate, its p99 and max, and what happens when it overruns.


## 4. Measurement traps

Every one of these silently changes the number by tens of percent or more.

| Trap | What goes wrong | What to do |
|---|---|---|
| **Debug build** | `-O0` vs `-O3` is routinely **10–50×** on Eigen/OpenCV-heavy code, because expression templates and inlining are what make Eigen fast at all. `RelWithDebInfo` is not `Release` if `NDEBUG` is unset — Eigen's bounds assertions stay live. | State `CMAKE_BUILD_TYPE`, and check `NDEBUG` is defined. |
| **Playback rate** | Running a rosbag at `--rate 0.2` guarantees no frame is ever dropped, so a system that cannot keep up in real time looks fine — **and produces a different, better trajectory**. An offline number is not a real-time number, and an offline *accuracy* number obtained this way is not a real-time accuracy number either. | Report accuracy at `--rate 1.0` too, and report the drop count. |
| **Wall clock vs CPU time** | A 4-thread system can burn 4× more CPU-seconds than wall-seconds. Reporting wall clock alone hides that it needs 4 free cores. | Report both, plus thread count and core count. |
| **I/O and decode excluded** | Papers usually time from decoded image in memory. Deployments pay demosaic, undistort, and transport. | State the boundaries of what is timed. |
| **Frequency scaling / thermal throttle** | A laptop at boost clock for the first 60 s then throttling 30–40 % gives a first-run number nobody can reproduce. | `cpupower frequency-set -g performance`, disable turbo for reproducibility, run long enough to reach steady state, and report the steady-state number. |
| **Cold start** | Page faults, allocator warm-up, GPU kernel JIT, vocabulary/BoW loading. The first 50–100 frames are not representative. | Discard a warm-up window and say how long it was. |
| **Timer choice** | `std::chrono::system_clock` is wall time and can jump (NTP). | `steady_clock` for durations. |
| **Instrumentation overhead** | Per-stage scoped timers with locked logging can cost more than the stage on a 10 ms budget. | Lock-free ring buffer, or measure the measurement. |
| **Run-to-run variance** | Timing is at least as noisy as accuracy — everything in [benchmark methodology §1](benchmark_methodology.ipynb) applies. | Multiple runs, report the distribution. |


## 5. Memory and energy — the constraints that actually kill deployments

On an embedded target these bind before compute does, and neither appears in a
typical results table.

**Peak RSS, not average.** A system that peaks at 3.5 GB during global BA does
not run on a 4 GB Jetson, whatever its average is. The OOM killer does not care
about your mean.

**Growth is the real question.** Most SLAM systems have an unbounded map: every
keyframe, descriptor, and landmark is retained. That is fine for a 2-minute
EuRoC sequence and fatal for an 8-hour deployment. Report **RSS versus time**,
not one number, and state whether the map is bounded and by what mechanism
(keyframe culling, submap sliding window, descriptor pruning).

$$
\text{peak RSS: } \texttt{VmHWM} \text{ in } \texttt{/proc/<pid>/status}
$$

**Energy.** On a battery platform the currency is **joules per frame**, not
seconds per frame. A GPU front-end that halves latency while tripling power can
be the wrong trade on a drone where flight time is the binding constraint.
Measure with `tegrastats` (Jetson), RAPL via `perf stat -e power/energy-pkg/`
(Intel), or an inline power meter.

**Determinism under load.** A number measured on an idle desktop says nothing
about behaviour when the planner, the perception stack, and the logger are
competing for the same cores. If the target is a shared platform, measure under
representative load.


## 6. Tooling

None of the trajectory-evaluation toolboxes measure any of this — `evo` and
`rpg_trajectory_evaluation` consume poses and nothing else, the same blind spot
they have for covariance
([estimator consistency §7](estimator_consistency.ipynb)).

| Need | Tool |
|---|---|
| Whole-process time, cache, branch stats | `perf stat -d ./prog` |
| Where the time goes | `perf record -g` + `perf report`, or `hotspot` |
| Per-stage, frame-by-frame, with a timeline | [Tracy](https://github.com/wolfpld/tracy) — nanosecond scoped zones, live view; the right tool for a 10 ms budget |
| Lightweight in-code timing | `std::chrono::steady_clock` scoped timers into a lock-free ring buffer, dumped at exit |
| Peak memory | `VmHWM` from `/proc/<pid>/status`; `/usr/bin/time -v`; `heaptrack` for allocation profiles |
| ROS end-to-end latency | `ros2 topic delay /topic` (uses the header stamp — this is the §1 definition of latency), `ros2 topic hz`, `rqt_top` |
| ROS message drops | Subscriber queue depth and QoS; see [ROS 2 QoS](../ros2_qos.md) |
| GPU | `nvidia-smi dmon`, Nsight Systems; `tegrastats` on Jetson |
| Energy | `perf stat -e power/energy-pkg/`, `tegrastats`, inline power meter |

**A note on `ros2 topic hz`:** it measures *publication* rate, which is the
throughput of §1, not the latency. A node publishing at a steady 30 Hz with a
300 ms internal pipeline looks perfect to `hz` and is unusable for control.
`ros2 topic delay` is the one that answers the real question, and only if the
header stamp is the *sensor* stamp rather than the publish time — a very common
bug that makes latency measure as ~0.


## 7. Reporting template

**Platform** — CPU model and core count, RAM, GPU, OS, compiler and version,
`CMAKE_BUILD_TYPE`, whether `-march=native`, thread count used, CPU governor.
A timing number without this is not interpretable.

**Per-thread budget table:**

| Thread | Rate | mean | p50 | p99 | max | overrun consequence |
|---|---|---|---|---|---|---|
| IMU propagation | 200 Hz | | | | | state output stops |
| Front-end | 20 Hz | | | | | frames dropped |
| Local BA | 4 Hz | | | | | map staleness |
| Loop closure | ~0.1 Hz | | | | | late correction |

**End-to-end:** sensor-timestamp-to-pose latency (p50 / p99 / max), sustained
throughput, RTF, and the frame-drop count at `--rate 1.0`.

**Resources:** peak RSS, RSS-vs-time plot, peak CPU utilisation (as a fraction
of total cores), J/frame if the target is battery powered.

**Statistics:** multiple runs, warm-up discarded and how much, timing
distribution not just the mean — per
[benchmark methodology §2](benchmark_methodology.ipynb).

**The pairing that matters:** report accuracy and runtime **from the same
runs**, at `--rate 1.0`. An accuracy number from an offline run and a timing
number from a real-time run describe two different systems, and combining them
in one table is the most common way a real-time claim goes wrong.

## 8. See also

* [Trajectory analysis](trajectory_analysis.ipynb) — accuracy metrics
* [Evaluation metrics for poses and trajectories](../vo_evaluation_metrics.ipynb) — ATE/RPE vs mAA
* [Estimator consistency](estimator_consistency.ipynb) — is the covariance honest
* [Benchmark methodology](benchmark_methodology.ipynb) — variance and significance, which apply to every number here
* [Robotic system design](../robotic_system_design.md) · [ROS 2 QoS](../ros2_qos.md)
* [Head-to-head VIO comparison](../../vio_benchmark/docs/COMPARISON.md) · [Parameter reference](../../vio_benchmark/docs/PARAMETERS.md)
